# 01 - CFPB Consumer Complaint Dataset Exploration & Preprocessing

This notebook provides exploratory data analysis and text preprocessing inspection for the **Customer Complaint Similarity & Categorisation** project using `src.data_loader` and `src.preprocessing`.

### Objectives:
1. Load the CFPB dataset using `src.data_loader.load_dataset`.
2. Inspect dataset dimensions (shape) and complete column schema.
3. View sample records across standardized fields (`complaint_id`, `category`, `text`).
4. Audit missing values across all dataset fields.
5. Quantify unique financial product categories and their distribution.
6. Report the total number of usable complaint narrative records.
7. Demonstrate the classical text preprocessing pipeline (cleaning, tokenization, stopword removal) on real complaint narratives.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_dataset, get_dataset_summary, resolve_columns
from src.preprocessing import clean_text, tokenize, remove_stopwords, preprocess_text, preprocess_series
print("Modules src.data_loader and src.preprocessing successfully imported.")

## 1. Load Dataset & Structural Overview
We load the dataset using `load_dataset()`. The loader automatically detects CFPB column variations, preserves original columns, and standardizes key fields to `text`, `category`, and `complaint_id`.

In [ ]:
data_path = project_root / "data" / "complaints.csv"
df = load_dataset(data_path, drop_invalid=False, standardize_columns=True)
summary = get_dataset_summary(df)

print(f"Dataset Shape: {summary['total_rows']} rows, {summary['total_columns']} columns")
print(f"Resolved Text Column:     {summary['text_column']}")
print(f"Resolved Category Column: {summary['category_column']}")
print(f"Resolved ID Column:       {summary['id_column']}")

## 2. Complete Column Names

In [ ]:
print(f"Total columns ({len(df.columns)}):\n")
for i, col in enumerate(df.columns):
    print(f"  [{i:2d}] {col}")

## 3. First Five Rows of Relevant Fields

In [ ]:
relevant_cols = ["complaint_id", "category", "text"]
df_sample = df[relevant_cols].head(5)
for idx, row in df_sample.iterrows():
    print(f"Record #{idx + 1} | ID: {row['complaint_id']} | Category: {row['category']}")
    print(f"Narrative: {str(row['text'])[:120]}...\n")

## 4. Missing Values Audit

In [ ]:
missing_series = df.isnull().sum()
missing_df = pd.DataFrame({
    "Missing Count": missing_series,
    "Missing Percentage (%)": (missing_series / len(df) * 100).round(2)
})
missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values(by="Missing Count", ascending=False)
print("Columns with missing values:")
print(missing_df)

print(f"\nMissing Complaint Narratives: {summary['missing_text_count']}")
print(f"Missing Product Categories:  {summary['missing_category_count']}")

## 5. Unique Product Categories & Distribution

In [ ]:
category_counts = df["category"].value_counts()
print(f"Number of Unique Product Categories: {summary['num_unique_categories']}\n")
print("Top Categories by Count:")
print(category_counts)

# Visual plot of top categories
plt.figure(figsize=(10, 6))
sns.barplot(
    x=category_counts.head(10).values,
    y=category_counts.head(10).index,
    hue=category_counts.head(10).index,
    palette="crest",
    legend=False
)
plt.title("Top 10 Consumer Complaint Product Categories")
plt.xlabel("Number of Complaints")
plt.ylabel("Product Category")
plt.tight_layout()
plt.show()

## 6. Usable Records Summary

In [ ]:
print(f"Total Records in Dataset:     {summary['total_rows']:,}")
print(f"Usable Narrative Records:     {summary['usable_rows_count']:,} ({summary['usable_rows_count']/summary['total_rows']*100:.2f}%)")
print(f"Invalid / Blank Records:      {summary['total_rows'] - summary['usable_rows_count']:,}")

## 7. Text Preprocessing Pipeline Demonstration

The classical text preprocessing pipeline transforms noisy, raw consumer complaint text into a clean standardized format ready for vectorization.

**Pipeline Architecture:**
$$\text{Raw Complaint} \rightarrow \text{Lowercase} \rightarrow \text{Noise / URL / Email Removal} \rightarrow \text{Redaction Stripping (\\bx\{2,\}\\b)} \rightarrow \text{Number Retention} \rightarrow \text{Stopword Filtering} \rightarrow \text{Clean Text}$$

**Design Decision on Numbers:** Numeric tokens (e.g. dollar amounts, years, percentages) are intentionally retained as they carry discriminative financial meaning (e.g. loan origination years or dispute amounts).

In [ ]:
# Preprocess a sample slice of records
sample_slice = df[["complaint_id", "category", "text"]].head(5).copy()
sample_slice["cleaned_text"] = preprocess_series(sample_slice["text"])
sample_slice["raw_chars"] = sample_slice["text"].str.len()
sample_slice["clean_chars"] = sample_slice["cleaned_text"].str.len()
sample_slice["raw_words"] = sample_slice["text"].apply(lambda x: len(str(x).split()))
sample_slice["clean_words"] = sample_slice["cleaned_text"].apply(lambda x: len(str(x).split()))

print("=== Real Complaint Preprocessing Comparisons ===\n")
for idx, row in sample_slice.iterrows():
    print(f"Complaint ID: {row['complaint_id']} | Category: {row['category']}")
    print(f"  [RAW]   ({row['raw_chars']} chars, {row['raw_words']} words):\n    {str(row['text'])[:130]}...")
    print(f"  [CLEAN] ({row['clean_chars']} chars, {row['clean_words']} words):\n    {str(row['cleaned_text'])[:130]}...")
    reduction = (1 - row['clean_chars'] / row['raw_chars']) * 100
    print(f"  [NOISE REDUCTION]: {reduction:.1f}% reduction in character volume\n")

In [ ]:
# Display summary table of before vs. after word counts
sample_slice[["complaint_id", "category", "raw_words", "clean_words", "raw_chars", "clean_chars"]]